# Pull Holdings and Derivative Data from Eagle

How to wait until Element is Visible in Selenium Python  

https://pythonexamples.org/python-selenium-wait-until-element-is-visible/

In [1]:
print('\n\n################################################')
print('#                                              #')
print('#   START 1/4 derv_checker_downloading.ipynb   #')
print('#                                              #')
print('################################################\n\n')



################################################
#                                              #
#   START 1/4 derv_checker_downloading.ipynb   #
#                                              #
################################################




In [2]:
# load libarries and set up paths

import time
start_time = time.time()
start_time_derivative_downloading = time.time()
print("Importing libraries and setting paths for derv_checker_downloading.ipynb ...")

# load libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime
from constants import pthPy, pthEXPORTS, pth_dl, pthLOCAL
from utilities import timediff, osprey, prior_working_day

fund_load = 200  # max number of funds to load in one go from osprey;
# if more than this, the list of funds is split into two halves
# and loaded in two separate calls to osprey() with the
# "half1" and "half2" options, and then the two halves #
# are combined into one dataframe and saved as a csv file in the Downloads folder

# get report date and selected summary sheet option
df1 = pd.read_excel(pthPy, sheet_name="arc", header=None, usecols="A").dropna()
full = df1[0].iloc[1:]
funds = (",").join(full.tolist())

df2 = pd.read_excel(pthPy, sheet_name="arc", header=None, usecols="E").dropna()
k = df2.iloc[1, 0]
rptDate = (
    k.date() if isinstance(k, datetime) else prior_working_day(datetime.today()).date()
)  # prior working day or report date override; has type datetime()
summ_yn = df2.iloc[3, 0]

print(f" {timediff(start_time, time.time())} importing libraries and setting paths for derv_checker_downloading.ipynb","\n")

Importing libraries and setting paths for derv_checker_downloading.ipynb ...
 24.6sec importing libraries and setting paths for derv_checker_downloading.ipynb 



In [3]:
# derive file names
start_time = time.time()
print("Deriving file names ...")

filename = os.path.join(pthEXPORTS, f'Derv {rptDate.strftime("%d%b%Y")}.xlsx')
fPARN    = os.path.join(Path.home(), "Downloads", f"PARN ({len(full)}) {rptDate.strftime('%d%b%Y')}.csv")
fDE      = os.path.join(Path.home(), "Downloads", f"DERV ({len(full)}) {rptDate.strftime('%d%b%Y')}.csv")
hlf  = int(len(full)/2)
half1 = (',').join(full[:hlf])
half1_name = f'PARN half1({len(full[:hlf])}) {rptDate.strftime("%d%b%Y")}.csv'
half2 = (',').join(full[hlf:])
half2_name = f'PARN half2({len(full[hlf:])}) {rptDate.strftime("%d%b%Y")}.csv'
full_name = f'PARN ({len(full)}) {rptDate.strftime("%d%b%Y")}.csv'

s = "s" if len(full) > 1 else ""
print(f" \n{rptDate.strftime('%A %d %b %Y')} for {len(full)} fund{s}:\n {funds}\n")
print(f" {'No' if summ_yn == 'No' else 'A'} summary sheet is required", "\n")

print(f"{timediff(start_time, time.time())} deriving file names","\n")

Deriving file names ...
 
Friday 03 Jul 2026 for 151 funds:
 3BBCIINC,ABMMBND,ADRRC,ADVMB,AFGBAL,AFLBAL,AFLMMF,AMMARF,AMPBQP,ASBTOS,ASHFLX,BCICEF,BCIFIF,BPROV,CCNPF,CMPFCASH,CMPFFLEX,CMPFINC,CSIRBQP,ECICBALC,ELCIPF,ENGENIP,ENGENMBF,FEMPBF,GACASH,GAEMBF,GEMSMEDC,GMRETF,GMRETF2,GRFINV,GTCWP2,HOLADC,HOLDINC,HOLEQU,HOLIDC,HOLMMF,HOLYPF,HOSMED,IJGCOR,IJGIPF,IMPALA,IMPBAL_C,IMPREF_C,IPIPF,ISPFP,LEZAFFI,LEZALDI,LPIIFTAA,MASAINC,MASAINRF,MASAMMF,MASASI,MASASIRF,MEDINC,MGFYQP,MOMBBF,MOMBBRF,MOMFLX,MOMFXB,MOMIPF,MOMPRET,MOMTAAHI,MOMTAALI,MOMTAAMI,MULTICH,MWPFEQU,MWPFILB,MYQIP,NEDMED,NESEQU,NFMWEQU,NFMWAGG,NGKINC,OMMAIF,PABS,PBFETF,PBNDQ,PCBF,PCCEF,PCEQTF,PCGEARF,PCGEF,PCSHQ,PEQ,PEQF,PETFIP,PEYF,PFFIF,PGCBF,PGCEF,PGIPFA,PGPCEM_C,PGPCGE_C,PGPGARF,PGPGBF_C,PGPGIF_C,PGPRF_C,PICPROV,PIF,PIMBAL,PIMEVO,PIMIDF,PIPF,PIPFP,PLMED,PLPRNA,PMMF,POIF_C,PORFIP,PPEF,PPOS,PPSBAL_C,PPSFLEX,PPSNAM,PPSRBNQ,PPWEQU,PRPABF,PRPDBF,PSIF,PSILB,PSPAM,PSSPFEQU,PSTIF,PTIF,QIFFGIF,SAAMCAU,SAAMINC,SAAMMOD,SABCCSHF,SDINCPM,SILA

In [4]:
# download fund holdings

start_time = time.time()
print("Downloading and then saving holdings data ...")

if len(full) > fund_load: # if more than 200 funds are in the list ...
    # ... get holdings for the first half of funds in the list
    start_time_1 = time.time()
    print("  ... downloading first of two subsets of holdings")
    # check if the file was already downloaded before running osprey()
    # if os.path.isfile(pth_dl + r'\\' + half1_name):
    if os.path.exists(os.path.join(pth_dl, half1_name)) and os.path.getsize(os.path.join(pth_dl, half1_name)) > 0:
        print(f"  {half1_name} already exists\n")
        pass
    else:
        osprey("parn", half1, rptDate, rptDate, "half1", "csv")
        print(f"  {timediff(start_time_1, time.time())} downloading first of two subsets of holdings")

    # ... get holdings for the second half of funds in the list
    start_time_2 = time.time()
    print("  ... downloading second of two subsets of holdings")
    # check if the file was already downloaded before running osprey()
    # if os.path.isfile(pth_dl + r'\\' + half2_name):
    if os.path.exists(os.path.join(pth_dl, half2_name)) and os.path.getsize(os.path.join(pth_dl, half2_name)) > 0:
        print(f"  {half2_name} already exists\n")
        pass
    else:
        osprey("parn", half2, rptDate, rptDate, "half2", "csv")
        print(f"  {timediff(start_time_2, time.time())} downloading second of two subsets of holdings")

    # combine dataframes of the two half sets of holdings https://pandas.pydata.org/docs/user_guide/merging.html
    df1 = pd.read_csv(os.path.join(pth_dl, half1_name))
    df2 = pd.read_csv(os.path.join(pth_dl, half2_name))
    df3 = pd.concat([df1, df2])
    # print(len(df1), len(df2), len(df1) + len(df2), len(df3))
    
    # write the combined dataframe to a csv file in the Downloads folder
    df3.to_csv(os.path.join(pth_dl, f'PARN ({len(full[hlf:]) + len(full[:hlf])}) {rptDate.strftime("%d%b%Y")}.csv'), index = False)

else: # else get all the holdings in one go
    # check if the file was already downloaded before running osprey()
    # if os.path.isfile(pth_dl + r'\\' + full_name):
    if os.path.exists(os.path.join(pth_dl, full_name)) and os.path.getsize(os.path.join(pth_dl, full_name)) > 0:
        print(f"  {full_name} already exists\n")
        pass
    else:
        start_time_h = time.time()
        osprey("parn", funds, rptDate, rptDate, "", "csv")
        print(f"  {timediff(start_time_h, time.time())} downloading all holdings")
    print('', full_name, '\n')

print(f"\n{timediff(start_time, time.time())} downloading and then saving holdings data","\n")

(2) loading libraries
Expected file name based on input values:
   PARN (151) 03Jul2026.csv
which does not yet exist in the Downloads folder.

(2a) Checking if rpt_type is 'fnav', in which case, replacing '_C' in fund name
(3) setting report suffix for .xls vs .csv
(4) Importing the webdriver (attempt 1/5)
(5) Starting the web driver and opening the browser ...
(6) Entering credentials to open the browser on the default web page...
(7) having logged in, opening the selected report page ...
(7a) testing for the presence of an alert and accepting it if it exists
(8) switching to the query page ...
(9) updating the FROM calendar ...
(10) updating the TO calendar, if it exists ...
(11) getting the web element for the FUND LIST and assigning values to it ...
(12) clicking 'Entity ID' to trigger the report generation ...
(12a) testing for the presence of an authentication     alert after clicking Submit ...
Exception in steps 7a onwards: Message: User prompt of type promptUserAndPass is not 

In [5]:
# download derivative metrics

start_time = time.time()
print('Downloading and then saving derivative data ...')

derv_name = f'DERV ({len(full)}) {rptDate.strftime("%d%b%Y")}.csv'
# check if the file was already downloaded before running osprey()
if os.path.isfile(os.path.join(pth_dl, derv_name)):
    print(f"  {derv_name} already exists")
    pass
else:
    start_time_d = time.time()
    osprey("derv", funds, rptDate, rptDate, "", "csv")
    print(f"  {timediff(start_time_d, time.time())} downloading derivative data")

print(f'{timediff(start_time, time.time())} downloading and then saving derivative data','\n')

(2) loading libraries
Expected file name based on input values:
   DERV (151) 03Jul2026.csv
which does not yet exist in the Downloads folder.

(2a) Checking if rpt_type is 'fnav', in which case, replacing '_C' in fund name
(3) setting report suffix for .xls vs .csv
(4) Importing the webdriver (attempt 1/5)
(5) Starting the web driver and opening the browser ...
(6) Entering credentials to open the browser on the default web page...
(7) having logged in, opening the selected report page ...
(7a) testing for the presence of an alert and accepting it if it exists
(8) switching to the query page ...
(9) updating the FROM calendar ...
(10) updating the TO calendar, if it exists ...
(11) getting the web element for the FUND LIST and assigning values to it ...
(12) clicking 'Entity ID' to trigger the report generation ...
(12a) testing for the presence of an authentication     alert after clicking Submit ...
(13) getting the web element of the 'Submit' button and then clicking it ...
(13a) te

In [6]:
# get paths to the holdings and derivative metric files
print(f"Getting the reporting date and latest downloaded holdings and derivatives files","\n")

print(" Expected downloads:")
print(f"  {fPARN} which {'exists' if os.path.exists(fPARN) else 'does not exist'}")
print(f"  {fDE} which {'exists' if os.path.exists(fDE) else 'does not exist'}")
print(f" A summary sheet is{' not' if summ_yn == 'No' else ''} required", "\n")
print(f"{timediff(start_time, time.time())} getting the reporting date and latest downloaded holdings and derivatives files","\n")

Getting the reporting date and latest downloaded holdings and derivatives files 

 Expected downloads:
  C:\Users\hilton.netta\Downloads\PARN (151) 03Jul2026.csv which exists
  C:\Users\hilton.netta\Downloads\DERV (151) 03Jul2026.csv which exists
 A summary sheet is required 

1min 31.6sec getting the reporting date and latest downloaded holdings and derivatives files 



In [7]:
print(f'{timediff(start_time_derivative_downloading, time.time())} downloading holdings and derivative metrics. Next step is compiling.', '\n')

5min 39.0sec downloading holdings and derivative metrics. Next step is compiling. 



In [8]:
print('\n\n################################################')
print('#                                              #')
print('#    END 1/4 derv_checker_downloading.ipynb    #')
print('#                                              #')
print('################################################\n\n')



################################################
#                                              #
#    END 1/4 derv_checker_downloading.ipynb    #
#                                              #
################################################


